# Finike Portakal Fiyat Tahmini — Keşifsel Veri Analizi (EDA)

Bu notebook, toplanan verileri analiz eder ve model geliştirme için içgörüler sağlar.

## Veri Kaynakları
1. **Hal Fiyatları** — İBB İstanbul Hal günlük portakal fiyatları (2007-2026)
2. **Hava Durumu** — Open-Meteo Finike bölgesi (sıcaklık, yağış, don)
3. **NDVI/Uydu** — Bitki örtüsü sağlığı (sentetik, Sentinel-2 bazlı)
4. **Döviz Kurları** — USD/TRY ve rakip ülke paraları
5. **Yabancı Pazarlar** — FAO endeksi, AB fiyatları, rakip üretim
6. **Politika/Haber** — İhracat yasakları, don olayları, ekonomik şoklar

## Adımlar
1. Veri yükleme ve kontrol
2. Fiyat analizi ve mevsimsellik
3. Hava durumu analizi ve don olayları
4. Döviz kuru ve yabancı pazar analizi
5. Politika/olay etki analizi
6. Korelasyon analizi
7. Model sonuçları

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print('Setup complete!')

## 1. Veri Toplama

Eğer ham veriler yoksa, pipeline'ı çalıştırarak verileri toplayın.

In [ ]:
# Veri toplama (ilk seferde çalıştırın)
from src.pipeline import collect_data

# Eğer veriler yoksa:
# collect_data(start_year=2018)

## 2. Fiyat Analizi

In [ ]:
# Hal fiyatlarını yükle
prices = pd.read_csv(RAW_DIR / 'hal_prices.csv', parse_dates=['date'])
print(f'Toplam kayıt: {len(prices)}')
print(f'Tarih aralığı: {prices["date"].min().date()} — {prices["date"].max().date()}')
print(f'Fiyat aralığı: {prices["min_price"].min():.2f} — {prices["max_price"].max():.2f} TL/kg')

if 'avg_price' not in prices.columns:
    prices['avg_price'] = (prices['min_price'] + prices['max_price']) / 2

display(prices.describe())
prices.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

# 1. Günlük fiyat trendi
ax1 = axes[0]
ax1.fill_between(prices['date'], prices['min_price'], prices['max_price'], alpha=0.3, color='orange', label='Min-Max aralığı')
ax1.plot(prices['date'], prices['avg_price'], color='darkorange', linewidth=0.8, label='Ortalama')
ax1.set_title('İstanbul Hal — Portakal Günlük Fiyatı (TL/kg)', fontsize=14)
ax1.set_ylabel('Fiyat (TL/kg)')
ax1.legend()

# 2. Yıllık ortalama
ax2 = axes[1]
yearly = prices.set_index('date').resample('YE')['avg_price'].mean()
yearly.plot(kind='bar', ax=ax2, color='orange', alpha=0.8)
ax2.set_title('Yıllık Ortalama Portakal Fiyatı', fontsize=14)
ax2.set_ylabel('TL/kg')
ax2.set_xticklabels([d.strftime('%Y') for d in yearly.index], rotation=45)

# 3. Mevsimsellik — ay bazında kutu grafiği
ax3 = axes[2]
prices['month'] = prices['date'].dt.month
month_names = {1:'Oca',2:'Şub',3:'Mar',4:'Nis',5:'May',6:'Haz',7:'Tem',8:'Ağu',9:'Eyl',10:'Eki',11:'Kas',12:'Ara'}
prices['month_name'] = prices['month'].map(month_names)
# Use recent years for seasonality (2020+)
recent = prices[prices['date'].dt.year >= 2020]
recent.boxplot(column='avg_price', by='month', ax=ax3)
ax3.set_title('Aylık Fiyat Dağılımı (2020-2026)', fontsize=14)
ax3.set_xlabel('Ay')
ax3.set_ylabel('TL/kg')
plt.suptitle('')

plt.tight_layout()
plt.show()

# Mevsimsellik özeti
print('\nMevsimsel ortalamalar (2020-2026):')
print(recent.groupby('month')['avg_price'].agg(['mean','std','min','max']).round(2))

## 3. Hava Durumu Analizi

In [ ]:
# Hava durumu verisi
weather_path = RAW_DIR / 'weather_finike.csv'

if weather_path.exists():
    weather = pd.read_csv(weather_path, parse_dates=['date'])
    print(f'Toplam kayıt: {len(weather)}')
    print(f'Tarih aralığı: {weather["date"].min()} - {weather["date"].max()}')
    display(weather.describe())
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    
    # Sıcaklık
    axes[0].fill_between(weather['date'], weather['temp_min'], weather['temp_max'], alpha=0.3, color='red')
    axes[0].plot(weather['date'], weather['temp_mean'], color='red', linewidth=0.5)
    axes[0].axhline(y=0, color='blue', linestyle='--', alpha=0.5, label='Don sınırı (0°C)')
    axes[0].axhline(y=-5, color='darkblue', linestyle='--', alpha=0.5, label='Şiddetli don (-5°C)')
    axes[0].set_title('Finike Sıcaklık (°C)', fontsize=14)
    axes[0].legend()
    
    # Yağış
    axes[1].bar(weather['date'], weather['precipitation'], color='blue', alpha=0.5, width=1)
    axes[1].set_title('Günlük Yağış (mm)', fontsize=14)
    
    # Don günleri (aylık)
    if 'frost' in weather.columns:
        monthly_frost = weather.set_index('date').resample('ME')['frost'].sum()
        monthly_frost.plot(kind='bar', ax=axes[2], color='darkblue')
        axes[2].set_title('Aylık Don Günü Sayısı', fontsize=14)
    
    plt.tight_layout()
    plt.show()
else:
    print('Hava durumu verisi bulunamadı.')

## 4. NDVI / Uydu Analizi

In [ ]:
# NDVI verisi
ndvi_path = RAW_DIR / 'ndvi_finike.csv'

if ndvi_path.exists():
    ndvi = pd.read_csv(ndvi_path, parse_dates=['date'])
    print(f'Toplam gözlem: {len(ndvi)}')
    display(ndvi.describe())
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    axes[0].plot(ndvi['date'], ndvi['ndvi_mean'], color='green', marker='o', markersize=3)
    axes[0].fill_between(ndvi['date'], 
                         ndvi['ndvi_mean'] - ndvi['ndvi_std'], 
                         ndvi['ndvi_mean'] + ndvi['ndvi_std'], 
                         alpha=0.2, color='green')
    axes[0].set_title('Finike NDVI Zaman Serisi', fontsize=14)
    axes[0].set_ylabel('NDVI')
    axes[0].axhline(y=0.3, color='orange', linestyle='--', label='Stres sınırı')
    axes[0].legend()
    
    # Aylık ortalama NDVI
    ndvi['month'] = ndvi['date'].dt.month
    monthly_ndvi = ndvi.groupby('month')['ndvi_mean'].mean()
    monthly_ndvi.plot(kind='bar', ax=axes[1], color='green', alpha=0.7)
    axes[1].set_title('Aylık Ortalama NDVI', fontsize=14)
    axes[1].set_xlabel('Ay')
    
    plt.tight_layout()
    plt.show()
else:
    print('NDVI verisi bulunamadı.')

## 5. Döviz Kuru Analizi

In [ ]:
# Döviz kurları
fx = pd.read_csv(RAW_DIR / 'fx_rates.csv', parse_dates=['date'])
print(f'FX kayıt: {len(fx)}, Tarih: {fx["date"].min().date()} — {fx["date"].max().date()}')

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

usd_col = 'TRY_per_USD' if 'TRY_per_USD' in fx.columns else 'USD_TRY'
axes[0].plot(fx['date'], fx[usd_col], color='purple', linewidth=1)
axes[0].set_title('USD/TRY Döviz Kuru', fontsize=14)
axes[0].set_ylabel('TL per USD')

# Portakal fiyatı vs USD/TRY (dual axis)
ax2 = axes[1]
ax2_twin = ax2.twinx()
monthly_prices = prices.set_index('date').resample('ME')['avg_price'].mean()
monthly_prices.plot(ax=ax2, color='orange', linewidth=1.5, label='Portakal (TL/kg)')
fx.set_index('date')[usd_col].resample('ME').mean().plot(ax=ax2_twin, color='purple', linewidth=1.5, label='USD/TRY')
ax2.set_ylabel('Portakal (TL/kg)', color='orange')
ax2_twin.set_ylabel('USD/TRY', color='purple')
ax2.set_title('Portakal Fiyatı vs Döviz Kuru', fontsize=14)
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 5. Yabancı Pazarlar ve Rekabet Analizi

In [ ]:
# Yabancı pazar verileri
foreign = pd.read_csv(RAW_DIR / 'foreign_markets.csv', parse_dates=['date'])
print(f'Foreign market kayıt: {len(foreign)}')
print(f'Sütunlar: {foreign.columns.tolist()}')

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# FAO Meyve Fiyat Endeksi
axes[0].plot(foreign['date'], foreign['fao_fruit_index'], color='darkgreen', linewidth=1.5)
axes[0].set_title('FAO Meyve Fiyat Endeksi (2014-2016=100)', fontsize=14)
axes[0].set_ylabel('Endeks')
axes[0].axhline(y=100, color='gray', linestyle='--', alpha=0.5)

# AB portakal fiyatları
if 'eu_orange_price_eur_100kg' in foreign.columns:
    axes[1].plot(foreign['date'], foreign['eu_orange_price_eur_100kg'], color='blue', linewidth=1.5)
    axes[1].set_title('AB Portakal Fiyatı (EUR/100kg) — İspanya', fontsize=14)
    axes[1].set_ylabel('EUR/100kg')

# Rekabet yoğunluğu
if 'competition_index' in foreign.columns:
    comp = foreign.dropna(subset=['competition_index'])
    axes[2].bar(comp['date'], comp['competition_index'], width=25, color='red', alpha=0.6)
    axes[2].set_title('Rakip İhracat Yoğunluğu Endeksi', fontsize=14)
    axes[2].set_ylabel('Rekabet Endeksi')

plt.tight_layout()
plt.show()

# Rakip üretim verileri
from src.data.foreign_markets import fetch_competitor_production
prod = fetch_competitor_production()
print('\nRakip Ülke Üretim (bin ton) — 2024:')
display(prod[prod['year'] == 2024][['country','production_kt','estimated_export_kt']].sort_values('production_kt', ascending=False))

In [ ]:
# Korelasyon analizi — hedef değişkenlerle en yüksek korelasyon
features = pd.read_csv(PROCESSED_DIR / 'feature_matrix.csv', parse_dates=['date'])
print(f'Özellik matrisi: {features.shape[0]} satır, {features.shape[1]} sütun')

numeric_cols = features.select_dtypes(include=[np.number]).columns

for horizon in [30, 60, 90]:
    target = f'target_{horizon}d'
    if target in features.columns:
        corr = features[numeric_cols].corrwith(features[target]).abs().sort_values(ascending=False)
        corr = corr.drop([c for c in corr.index if c.startswith('target_')])
        
        print(f'\n--- {horizon} Günlük Tahmin ile En Yüksek Korelasyonlar (Top 20) ---')
        print(corr.head(20).to_string())

# Korelasyon ısı haritası (top 25 feature)
top_features = features[numeric_cols].corrwith(features['target_30d']).abs().sort_values(ascending=False)
top_features = top_features.drop([c for c in top_features.index if c.startswith('target_')])
top_25 = top_features.head(25).index.tolist() + ['target_30d']

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(features[top_25].corr(), annot=True, fmt='.2f', cmap='RdYlBu_r', 
            center=0, ax=ax, annot_kws={'size': 7})
ax.set_title('En Önemli 25 Özellik — Korelasyon Matrisi', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Politika ve Olay Etki Analizi

In [ ]:
# Politika olayları ve fiyat etkisi
events = pd.read_csv(RAW_DIR / 'policy_events.csv', parse_dates=['date'])
policy_feat = pd.read_csv(RAW_DIR / 'policy_features.csv', parse_dates=['date'])

print(f'Toplam olay: {len(events)}')
print(f'\nOlay türleri:')
print(events['event_type'].value_counts().to_string())

# Olayları fiyat grafiği üzerine işaretle
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

ax1 = axes[0]
ax1.plot(prices['date'], prices['avg_price'], color='orange', linewidth=0.8, alpha=0.7)

colors = {'frost': 'blue', 'sanction': 'red', 'economic': 'purple', 
          'regulation': 'green', 'supply': 'brown', 'trade': 'gray', 'pandemic': 'black'}

for _, ev in events.iterrows():
    color = colors.get(ev['event_type'], 'gray')
    marker = '^' if ev['impact_direction'] == 'up' else 'v'
    ax1.axvline(x=ev['date'], color=color, alpha=0.3, linewidth=0.8)
    # Find price at event date
    price_at_date = prices.loc[prices['date'] >= ev['date'], 'avg_price']
    if not price_at_date.empty:
        ax1.scatter(ev['date'], price_at_date.iloc[0], color=color, marker=marker, 
                   s=ev['impact_magnitude'] * 50, zorder=5)

ax1.set_title('Portakal Fiyatı + Politika/Olay İşaretleri', fontsize=14)
ax1.set_ylabel('TL/kg')

# Legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color=c, label=t, linewidth=2) for t, c in colors.items()]
ax1.legend(handles=legend_elements, loc='upper left', fontsize=9)

# Politika etki skoru
ax2 = axes[1]
ax2.fill_between(policy_feat['date'], 0, policy_feat['policy_impact_score'], 
                 where=policy_feat['policy_impact_score'] > 0, color='red', alpha=0.4, label='Fiyat artırıcı')
ax2.fill_between(policy_feat['date'], 0, policy_feat['policy_impact_score'],
                 where=policy_feat['policy_impact_score'] < 0, color='blue', alpha=0.4, label='Fiyat düşürücü')
ax2.set_title('Politika Etki Skoru (pozitif = fiyat artışı)', fontsize=14)
ax2.set_ylabel('Etki Skoru')
ax2.legend()

plt.tight_layout()
plt.show()

# Olay listesi
print('\n--- Tüm Politika/Olay Kayıtları ---')
display(events[['date', 'event_type', 'description', 'impact_direction', 'impact_magnitude']])

## 7. Model Sonuçları

In [ ]:
# Model karşılaştırma sonuçları
results = pd.read_csv(PROCESSED_DIR / 'model_results.csv')
print('Model Karşılaştırma Tablosu:')
display(results)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, metric in enumerate(['MAE', 'MAPE', 'R2']):
    pivot = results.pivot(index='model', columns='horizon_days', values=metric)
    pivot.plot(kind='bar', ax=axes[i], rot=45)
    axes[i].set_title(metric, fontsize=14)
    axes[i].legend(title='Horizon (gün)')
    if metric == 'R2':
        axes[i].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    if metric == 'MAPE':
        axes[i].set_ylabel('MAPE (%)')

plt.suptitle('Model Performans Karşılaştırması', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# En iyi model
best_30d = results[results['horizon_days']==30].sort_values('MAE').iloc[0]
print(f'\nEn iyi 30 günlük model: {best_30d["model"]}')
print(f'  MAE: {best_30d["MAE"]:.2f} TL/kg')
print(f'  MAPE: {best_30d["MAPE"]*100:.1f}%')
print(f'  R²: {best_30d["R2"]:.3f}')